# 02 · Models and evaluation

Reproduces every Phase 4 table and figure from `data/processed/` (PLAN.md section 6):

| # | Experiment | Output |
|---|---|---|
| 1 | All models, developer-grouped 5-fold CV | `main_comparison.csv`, PR curves, confusion matrices |
| 2 | Temporal hold-out, best 3 models | `temporal_holdout.csv` |
| 3 | Ablation, drop one feature group | `ablation.csv`, `ablation.png` |
| 4 | Early detection (first 14 days) vs full | `early_detection.csv` |
| 5 | Class weights vs SMOTE | `imbalance_weights_vs_smote.csv` |
| 6 | Random vs grouped CV (leakage) | `leakage_random_vs_grouped.csv` |

The positive class is **unverified**. PR-AUC, ROC-AUC and recall at 5% FPR are threshold-free; precision, recall and F1 use a threshold fitted on the other folds (see `docs/deviations.md` #15). Every row is reported for **all** apps and for **organic** apps (found by search/similar, not via the RBI list), each with its base rate.

Set `N_ITER` (or the env var `LOANSHIELD_N_ITER`) to 30 for the final run; small values are for smoke tests only. MuRIL scores are included automatically if `results/oof/muril_grouped.csv` exists (see `muril_finetune.ipynb`).

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, HTML, display, Markdown
from loanshield.common import ROOT, path, cfg
os.chdir(ROOT)                      # all paths are relative to the repo root
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_columns", 30)
print("repo:", ROOT)

from loanshield.models import run_experiments as rx
from loanshield.models.splits import load_dataset
N_ITER = int(os.environ.get("LOANSHIELD_N_ITER", 30))
ds = load_dataset("features")
print(f"N_ITER={N_ITER} | {len(ds.y)} apps | {int(ds.y.sum())} unverified | base rate {ds.y.mean():.3f}")
if ds.y.sum() < 100:
    display(Markdown(f"> **Preliminary:** only {int(ds.y.sum())} unverified apps (plan target 100). Expect wide fold-to-fold variation."))
TABLE_COLS = ["model", "subset", "base_rate", "pr_auc", "roc_auc", "recall_at_5fpr", "precision", "recall", "f1", "macro_f1"]
def show(df, extra=()):
    cols = [c for c in [*extra, *TABLE_COLS] if c in df]
    display(df[cols].assign(base_rate=lambda d: d.base_rate.astype(float).round(3)) if "base_rate" in cols else df[cols])

## 1. Main comparison (grouped CV)

In [ ]:
main = rx.exp_main(ds, N_ITER)
show(main, ["family"])
display(Image(str(path("results", "figures", "pr_curves_grouped.png"))))
display(Image(str(path("results", "figures", "pr_curves_grouped_organic.png"))))
display(Image(str(path("results", "figures", "confusion_grouped.png"))))

## 2. Temporal hold-out
Train on apps released before the cut-off, test on later ones (best 3 single models from the main comparison).

In [ ]:
show(rx.exp_temporal(ds, N_ITER), ["cutoff", "n_pos_total"])

## 3. Ablation

In [ ]:
show(rx.exp_ablation(ds, N_ITER), ["dropped_group"])
display(Image(str(path("results", "figures", "ablation.png"))))

## 4. Early detection (first 14 days)
Snapshot-only metadata (installs, ratings, age, update date, developer counts) is dropped and reviews are limited to each app's first 14 days. Reviews were collected newest-first, so only apps released recently have early reviews (`docs/deviations.md` #13).

In [ ]:
show(rx.exp_early(ds, N_ITER), ["features"])

## 5. Class weights vs SMOTE

In [ ]:
show(rx.exp_smote(ds, N_ITER), ["imbalance"])

## 6. Leakage: random vs developer-grouped CV
A positive gap (random > grouped) means random splits flatter the model by letting one developer's near-identical apps sit on both sides.

In [ ]:
show(rx.exp_leakage(ds, N_ITER), ["split"])

## Run log
Every experiment above appended a row per model × split to `results/runs.csv`.

In [ ]:
runs = pd.read_csv(path("results", "runs.csv"))
display(runs.tail(15)[["timestamp", "experiment", "model", "split", "pr_auc", "roc_auc", "recall_at_5fpr"]])